In [ ]:
import sys
import os
import re
import json
import gc
from pathlib import Path
from tqdm import tqdm

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from workflow.config.ConfigNode import ConfigNode
from workflow.pointcloud import PointCloud as PC
from workflow.processor.converter.point_cloud_rasterization import point_cloud_rasterization

In [ ]:
'''把json文件中同一个边坡路段的多期巡飞点云转成图像，按 key 分文件夹存放。图片命名：时间戳+巡飞点云.png'''

yaml_file = '/home/gary/point-cloud-3d/params/CloudCompare.yaml'
json_file = '/home/gary/point-cloud-3d/output/可配对差分的点云名称/20260214学校操场实验.json'
save_root = '/home/gary/point-cloud-3d/output/可配对差分的点云图像/20260214学校操场实验'
cloud_root = '/home/gary/CloudPointProcessing/20260214学校操场实验'

cfg = ConfigNode.load_from_yaml(yaml_file)
cfg.validate()

json_file = Path(json_file)
save_root  = Path(save_root)
cloud_root = Path(cloud_root)
save_root.mkdir(parents=True, exist_ok=True)

TIMESTAMP_PATTERN = re.compile(r"(\d{12}_\d{1,3})")

with json_file.open("r", encoding="utf-8") as f:
    json_data = json.load(f)

for item in json_data.get("paired", []):
    key = item["key"]
    folders = item.get("folders", [])
    if not folders:
        continue

    # 创建该高速路段的输出文件夹
    key_dir = save_root / key
    key_dir.mkdir(exist_ok=True)

    for folder_name in folders:
        match = TIMESTAMP_PATTERN.search(folder_name)
        if not match:
            print(f"跳过无法提取时间戳: {folder_name}")
            continue
        
        timestamp = match.group(0)
        save_path = key_dir / f"{timestamp}_{key}.png"

        if save_path.exists():
            tqdm.write(f"跳过（图像已存在）: {save_path.name}")
            continue

        pc = PC(type='GPU')
        try:
            pc.load_from_las(cloud_root / folder_name / 'raw' / 'las' / 'cloud_merged.las')

            cloud_img = point_cloud_rasterization(
                pc=pc,
                sampling_rate=cfg.rasterization.sampling_rate,
                sampling_mode=cfg.rasterization.sampling_mode,
                enable_fill=False,
                enable_plt=False,
                fill_algorithm='nearest',
                reverse=cfg.rasterization.reverse
            )

            cloud_img.save(save_path)
            print(f"完成保存 “{folder_name}” 的点云图像")
        finally:
            del pc
            gc.collect()

print("所有可配对差分的点云已保存为图像")
